# Skin Disease — VGG19 + EfficientNet TTA Ensemble

**Workflow:**
1. Build test DataFrame from disk
2. Load each `.h5` model, run Test-Time Augmentation (TTA)
3. Compare 5 ensemble strategies
4. Report + confusion matrix for the best strategy
5. Per-class F1 grouped bar chart
6. Save best predictions

No training — inference only.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix

# ── Paths ──────────────────────────────────────────────────────────────────
VGG19_PATH  = 'best_skin_model_vgg19_enesmble.h5'
EFFNET_PATH = 'best_skin_model_effnet.h5'
TEST_DIR    = 'test'   # root folder — adjust if needed

# ── Hyper-parameters ───────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
TTA_PASSES = 7

class_names = ['Acne', 'Eczema', 'Fungal', 'Melanoma', 'Psoriasis', 'Vitiligo']

FOLDER_MAP = {
    'Acne'     : ['Acne and Rosacea Photos'],
    'Eczema'   : ['Eczema Photos', 'Atopic Dermatitis Photos'],
    'Psoriasis': ['Psoriasis pictures Lichen Planus and related diseases'],
    'Fungal'   : ['Tinea Ringworm Candidiasis and other Fungal Infections'],
    'Melanoma' : ['Melanoma Skin Cancer Nevi and Moles'],
    'Vitiligo' : ['Light Diseases and Disorders of Pigmentation'],
}

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

## 2. Build Test DataFrame

In [ ]:
def build_df(root_dir: str, folder_map: dict) -> pd.DataFrame:
    """
    Walk `root_dir`, map each sub-folder to a canonical class label,
    and return a DataFrame with columns ['filename', 'class'].

    Parameters
    ----------
    root_dir   : path to the dataset split directory (e.g. 'test')
    folder_map : dict mapping canonical label -> list of folder names on disk

    Returns
    -------
    pd.DataFrame with columns 'filename' (relative path) and 'class' (label str)
    """
    records = []
    folder_to_label = {}
    for label, folders in folder_map.items():
        for folder in folders:
            folder_to_label[folder] = label

    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue
        label = folder_to_label.get(folder_name)
        if label is None:
            print(f'  [WARN] Skipping unmapped folder: {folder_name}')
            continue
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
                records.append({
                    'filename': os.path.join(folder_name, fname),
                    'class'   : label,
                })

    df = pd.DataFrame(records)
    print(f"Test set: {len(df)} images  |  {df['class'].nunique()} classes")
    return df


df_test = build_df(TEST_DIR, FOLDER_MAP)

# Class distribution
fig, ax = plt.subplots(figsize=(8, 4))
df_test['class'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Test-Set Class Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

df_test.head()

## 3. TTA Helper Function

In [ ]:
def run_tta(model, df, directory, preprocess_fn, n_passes=7):
    """
    Run Test-Time Augmentation on a test set and return averaged softmax
    predictions together with ground-truth integer labels.

    Parameters
    ----------
    model        : loaded Keras model
    df           : DataFrame with 'filename' and 'class' columns
    directory    : root directory for relative filenames in df
    preprocess_fn: model-specific preprocessing function
    n_passes     : number of augmented forward passes

    Returns
    -------
    avg_preds : np.ndarray (N, n_classes) — mean softmax over TTA passes
    y_true    : np.ndarray (N,)           — integer ground-truth labels
    """
    tta_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=40,
        zoom_range=0.25,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest',
    )
    clean_datagen = ImageDataGenerator(preprocessing_function=preprocess_fn)

    clean_gen = clean_datagen.flow_from_dataframe(
        df, directory=directory,
        x_col='filename', y_col='class',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False,
    )
    y_true = clean_gen.classes

    all_passes = []
    for i in range(n_passes):
        tta_gen = tta_datagen.flow_from_dataframe(
            df, directory=directory,
            x_col='filename', y_col='class',
            target_size=IMG_SIZE, batch_size=BATCH_SIZE,
            class_mode='categorical', shuffle=False,
        )
        all_passes.append(model.predict(tta_gen, verbose=0))
        print(f'  Pass {i+1}/{n_passes} done', end='\r')

    print()
    return np.mean(all_passes, axis=0), y_true

## 4. Load Models and Run TTA

> Each model is deleted from memory after its TTA run to avoid OOM errors.

In [ ]:
# ── VGG19 ──────────────────────────────────────────────────────────────────
print("Loading VGG19 …")
vgg19_model = load_model(VGG19_PATH)
print(f"VGG19 loaded  |  params: {vgg19_model.count_params():,}")

print("\nRunning VGG19 TTA …")
vgg19_preds, y_true = run_tta(
    vgg19_model, df_test, TEST_DIR, vgg_preprocess, TTA_PASSES
)
np.save('vgg19_tta_preds.npy', vgg19_preds)

del vgg19_model
tf.keras.backend.clear_session()
print("VGG19 done  — saved → vgg19_tta_preds.npy")

In [ ]:
# ── EfficientNet ───────────────────────────────────────────────────────────
print("Loading EfficientNet …")
effnet_model = load_model(EFFNET_PATH)
print(f"EfficientNet loaded  |  params: {effnet_model.count_params():,}")

print("\nRunning EfficientNet TTA …")
effnet_preds, _ = run_tta(
    effnet_model, df_test, TEST_DIR, effnet_preprocess, TTA_PASSES
)
np.save('effnet_tta_preds.npy', effnet_preds)

del effnet_model
tf.keras.backend.clear_session()
print("EfficientNet done  — saved → effnet_tta_preds.npy")

# Ground truth
np.save('y_true.npy', y_true)

print(f"\nShapes — VGG19: {vgg19_preds.shape}  "
      f"EffNet: {effnet_preds.shape}  "
      f"y_true: {y_true.shape}")
assert vgg19_preds.shape == effnet_preds.shape, "Shape mismatch!"

## 5. Ensemble Strategies

In [ ]:
strategies = {
    'VGG19 TTA only'        : vgg19_preds,
    'EffNet TTA only'       : effnet_preds,
    'Simple Average (50/50)': (vgg19_preds + effnet_preds) / 2,
    'VGG19 30 / EffNet 70'  : (0.3 * vgg19_preds + 0.7 * effnet_preds),
    'VGG19 40 / EffNet 60'  : (0.4 * vgg19_preds + 0.6 * effnet_preds),
}

rows = []
for name, preds in strategies.items():
    r = classification_report(
        y_true, np.argmax(preds, axis=1),
        target_names=class_names, output_dict=True,
    )
    rows.append({
        'Strategy'   : name,
        'Accuracy'   : round(r['accuracy'], 4),
        'Macro F1'   : round(r['macro avg']['f1-score'], 4),
        'Weighted F1': round(r['weighted avg']['f1-score'], 4),
    })

results_df = pd.DataFrame(rows).set_index('Strategy')
display(results_df.style.highlight_max(color='lightgreen', axis=0))

best_name = results_df['Macro F1'].idxmax()
print(f"\nBest strategy (highest Macro F1): {best_name}")

## 6. Best Ensemble — Full Report + Confusion Matrix

In [ ]:
best_preds  = strategies[best_name]
y_pred_best = np.argmax(best_preds, axis=1)

print(f"Classification Report — {best_name}\n")
print(classification_report(y_true, y_pred_best, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names, ax=ax,
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix\n{best_name}', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_best.png', dpi=150)
plt.show()
print("Saved → confusion_matrix_best.png")

## 7. Per-Class F1 Grouped Bar Chart

In [ ]:
f1_data = {}
for name, preds in strategies.items():
    r = classification_report(
        y_true, np.argmax(preds, axis=1),
        target_names=class_names, output_dict=True,
    )
    f1_data[name] = [r[cls]['f1-score'] for cls in class_names]

f1_df = pd.DataFrame(f1_data, index=class_names)

x           = np.arange(len(class_names))
n_strategies = len(strategies)
bar_width   = 0.15
offsets     = np.linspace(
    -(n_strategies - 1) / 2 * bar_width,
     (n_strategies - 1) / 2 * bar_width,
    n_strategies,
)
colors = plt.cm.tab10(np.linspace(0, 0.5, n_strategies))

fig, ax = plt.subplots(figsize=(13, 6))
for idx, (name, color) in enumerate(zip(strategies.keys(), colors)):
    ax.bar(
        x + offsets[idx], f1_df[name],
        width=bar_width, label=name,
        color=color, edgecolor='white', linewidth=0.5,
    )

ax.set_xticks(x)
ax.set_xticklabels(class_names, fontsize=11)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Per-Class F1-Score by Ensemble Strategy', fontsize=14)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9, framealpha=0.85)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('per_class_f1_comparison.png', dpi=150)
plt.show()
print("Saved → per_class_f1_comparison.png")

## 8. Save Best Predictions

In [ ]:
np.save('best_ensemble_preds.npy', best_preds)
print(f"Saved → best_ensemble_preds.npy  (strategy: {best_name})")
print("\nAll artifacts:")
for fname in ['vgg19_tta_preds.npy', 'effnet_tta_preds.npy',
              'y_true.npy', 'best_ensemble_preds.npy',
              'confusion_matrix_best.png', 'per_class_f1_comparison.png']:
    exists = '✓' if os.path.exists(fname) else '✗'
    print(f"  {exists}  {fname}")